In [ ]:
import re
from pathlib import Path

import numpy as np
import polars as pl

In [ ]:
training_triples = Path("../dataset/training/train.triples.csv")
test_queries = Path("../dataset/baseline/queries.csv")
corpus = Path("../dataset/baseline/corpus.csv")

triples_df = pl.read_csv(training_triples)
queries_df = pl.read_csv(test_queries)
corpus_df = pl.read_csv(corpus)

### Check for lexical overlap between query and positive/negative passages in the training triples

In [ ]:
def tokenizer(text: str) -> list[str]:
    return re.findall(r"\w+", text)


def overlap(a, b):
    sa = set(tokenizer(a.lower()))
    sb = set(tokenizer(b.lower()))

    if not sa or not sb:
        return 0.0

    return len(sa & sb) / len(sa | sb)


bad = 0
total = 0

for row in triples_df.iter_rows(named=True):
    q = row["query"]
    pos = row["positive"]
    neg = row["negative"]
    op = overlap(q, pos)
    on = overlap(q, neg)
    if op <= on:
        bad += 1
    total += 1

print(
    f"Triples where lexical(q,pos) <= lexical(q,neg): {bad} / {total} = {bad / total:.1%}"
)

In [ ]:
def jaccard(a: str, b: str) -> float:
    tokena = set(tokenizer(a.lower()))
    tokenb = set(tokenizer(b.lower()))

    intersection = tokena.intersection(tokenb)
    union = tokena.union(tokenb)

    if len(union) == 0:
        return 0.0

    return len(intersection) / len(union)


pos_pairs = list(
    zip(
        triples_df["query"].to_list(),
        triples_df["positive"].to_list(),
    )
)

neg_pairs = list(
    zip(
        triples_df["query"].to_list(),
        triples_df["negative"].to_list(),
    )
)

pos_scores = [jaccard(a, b) for a, b in pos_pairs]
neg_scores = [jaccard(a, b) for a, b in neg_pairs]

print(f"Average Jaccard Positive: {np.mean(pos_scores)}")
print(f"Median Jaccard Positive: {np.median(pos_scores)}")
print(f"Average Jaccard Negative: {np.mean(neg_scores)}")
print(f"Median Jaccard Negative: {np.median(neg_scores)}")

### Check for Domain Shift using Adversarial Validation

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

train_queries = triples_df["query"].to_list()
test_queries = queries_df["query"].to_list()

overlap_queries = set(train_queries).intersection(set(test_queries))
print(f"Exact Overlap between Train and Test: {len(overlap_queries)} queries")

all_text = train_queries + test_queries
y = np.array([0] * len(train_queries) + [1] * len(test_queries))

print(f"Total samples: {len(all_text)}")
print(f"Training samples: {len(train_queries)}")
print(f"Test samples: {len(test_queries)}")

vectorizer = TfidfVectorizer(
    min_df=2,
    max_features=10000,
    ngram_range=(1, 2),  # Check unigrams and bigrams
)
X = vectorizer.fit_transform(all_text)

clf = LogisticRegression(solver="liblinear", random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(clf, X, y, cv=cv, scoring="roc_auc")

print("\n --- Adversarial Validation Results ---")
print(f"AUC Score: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")

if np.mean(scores) > 0.75:
    print("Strong Domain Shift detected!")
    print("The model can easily distinguish Train from Test queries.")

    # Check which features (words) are most indicative of the domain shift
    clf.fit(X, y)
    feature_names = vectorizer.get_feature_names_out()
    coefs = clf.coef_[0]

    # Top words indicating the test set
    top_test_indices = coefs.argsort()[-10:][::-1]
    print(
        "\nTop words specific to TEST set:",
        [feature_names[i] for i in top_test_indices],
    )

    # Top words indicating the training set
    top_train_indices = coefs.argsort()[:10]
    print(
        "Top words specific to TRAINING set:",
        [feature_names[i] for i in top_train_indices],
    )
elif np.mean(scores) >= 0.6:
    print("Moderate Domain Shift detected.")
    print(
        "There are some differences between the training and test sets, but they might not be the sole cause of failure."
    )
else:
    print("No significant Domain Shift.")
    print("The training and test sets look very similar statistically.")

### Check for Contradictions in the training triples

In [ ]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [
    tokenizer(doc)
    for doc in triples_df["positive"].to_list()
    + triples_df["negative"].to_list()
]

bm25 = BM25Okapi(tokenized_corpus)

contradictions = 0
n_samples = len(tokenized_corpus) // 2

for i, row in enumerate(triples_df.iter_rows(named=True)):
    query_tokens = tokenizer(row["query"])

    pos_id = i
    neg_id = i + n_samples

    scores = bm25.get_batch_scores(query_tokens, [pos_id, neg_id])

    pos_score = scores[0]
    neg_score = scores[1]

    if neg_score > pos_score:
        contradictions += 1

print(f"Contradiction Rate: {contradictions / n_samples:.2%}")
# Interpretation:
# > 20%: The dataset has significant False Negatives (The 'Negative' is often more relevant lexically).
# < 10%: The dataset is clean; the neural model just failed to learn.

### Check for granularity between training and test documents

In [ ]:
import numpy as np


def get_length(text):
    return len(tokenizer(text))


train_pos_lengths = triples_df["positive"].map_elements(get_length).to_list()
train_neg_lengths = triples_df["negative"].map_elements(get_length).to_list()
test_lengths = corpus_df["text"].map_elements(get_length).to_list()


print("\n--- Length Analysis (Word Count) ---")
print(
    f"{'Metric':<15} | {'Train Positive (Triplets)':<15} \t| {'Train Negative (Triplets)':<15} \t| {'Test (Corpus)':<15}"
)
print("-" * 100)
print(
    f"{'Mean':<15} | {np.mean(train_pos_lengths):<15.2f} \t\t| {np.mean(train_neg_lengths):<15.2f} \t\t| {np.mean(test_lengths):<15.2f}"
)
print(
    f"{'Median':<15} | {np.median(train_pos_lengths):<15.2f} \t\t| {np.median(train_neg_lengths):<15.2f} \t\t| {np.median(test_lengths):<15.2f}"
)
print(
    f"{'Max':<15} | {np.max(train_pos_lengths):<15} \t\t| {np.max(train_neg_lengths):<15} \t\t| {np.max(test_lengths):<15}"
)
print(
    f"{'Min':<15} | {np.min(train_pos_lengths):<15} \t\t| {np.min(train_neg_lengths):<15} \t\t| {np.min(test_lengths):<15}"
)


# Calculate how many Test documents are "Out of Distribution" for the model
train_pos_p95 = np.percentile(train_pos_lengths, 95)
train_neg_p95 = np.percentile(train_neg_lengths, 95)
test_pos_exceeding_train = np.sum(np.array(test_lengths) > train_pos_p95)
test_neg_exceeding_train = np.sum(np.array(test_lengths) > train_neg_p95)

print("\n--- Distribution Statistics ---")
print(
    f"95th Percentile of Training (Positive) Length: {train_pos_p95:.1f} words"
)
print(
    f"95th Percentile of Training (Negative) Length: {train_neg_p95:.1f} words"
)
print(
    f"Percentage of Test Documents (Positive) exceeding this length: {(test_pos_exceeding_train / len(test_lengths)) * 100:.1f}%"
)

print(
    f"Percentage of Test Documents (Negative) exceeding this length: {(test_neg_exceeding_train / len(test_lengths)) * 100:.1f}%"
)

In [ ]:
import altair as alt
import numpy as np

alt.data_transformers.enable("vegafusion")

df = pl.DataFrame(
    {
        "length": train_pos_lengths + train_neg_lengths + test_lengths,
        "group": ["Train (Positives)"] * len(train_pos_lengths)
        + ["Train (Negatives)"] * len(train_neg_lengths)
        + ["Test (Corpus Documents)"] * len(test_lengths),
    }
)

x_max = np.percentile(test_lengths, 95)
group_order = [
    "Train (Positives)",
    "Train (Negatives)",
    "Test (Corpus Documents)",
]

chart = (
    alt.Chart(df)
    .transform_density(
        "length",
        as_=["length", "density"],
        groupby=["group"],
        extent=[0, float(x_max)],
        steps=200,
    )
    .mark_area(opacity=0.8)
    .encode(
        x=alt.X("length:Q")
        .title("Length (Number of Token)")
        .scale(alt.Scale(domain=[0, float(x_max)])),
        y=alt.Y("density:Q").title("Density").stack(None),
        color=alt.Color("group:N")
        .sort(group_order)
        .legend(alt.Legend(title=None)),
    )
    .properties(
        width=600,
        height=400,
        # title="Distribution Shift: Training vs. Test Document Lengths",
    )
)

chart